# ChromaDB

In the previous notebook we compared **vector stores** and **vector databases**, and saw that **Chroma** is one of the tools that sits on both sides. Now we use it hands-on.

**Chroma** (also called ChromaDB) is an **open-source vector database** built for AI applications. It stores embeddings together with their text and metadata, and finds the most similar ones for a question. It is one of the most popular choices for learning and prototyping RAG, because:

- **No setup:** `pip install chromadb` and it runs inside your notebook. No server, no account, no API key.
- **It can embed for you:** pass plain text and Chroma creates the embeddings with a built-in model, or plug in your own (OpenAI, Hugging Face, ...).
- **Database features out of the box:** saving to disk, add / update / delete, and metadata filtering.
- **It grows with you:** the same code works in memory, on a local folder, against a server, or on Chroma Cloud.
- **Framework support:** LangChain and LlamaIndex both have Chroma integrations.

> **In one sentence:** Chroma is a vector database you can start as a simple Python library and later run as a server, without changing how you use it.

This notebook uses `chromadb` 1.5.x.

## The building blocks: client, collection, record

```
Client  (your connection to Chroma: memory, a local folder, or a server)
 │
 ├── Collection "research_papers"            ← like a table
 │     ├── record: id | document | embedding | metadata
 │     ├── record: id | document | embedding | metadata
 │     └── ...
 │
 └── Collection "hr_policies"
       └── ...
```

- **Client**: the entry point. The type of client you create decides **where the data lives** (next section).
- **Collection**: a named group of records, like a **table** in a SQL database. All records in a collection share the **same embedding model** and the **same distance metric**. Names must be 3 to 512 characters long, use only letters, digits, `.`, `_` and `-`, and start and end with a letter or digit.
- **Record**: the same four parts we saw in the previous notebook:

| Field | Argument in Chroma | What it holds |
|---|---|---|
| id | `ids` | A unique string key, e.g. `"chunk_042"`. **Required** |
| text | `documents` | The original chunk text |
| vector | `embeddings` | The embedding. Created for you if you only pass `documents` |
| metadata | `metadatas` | A flat dict of simple values (strings, numbers, booleans), e.g. `{"source": "1706.03762v7.pdf", "page": 3}` |

## Ways to run Chroma

| Client | Where the data lives | When to use it |
|---|---|---|
| `chromadb.EphemeralClient()` | In memory only; gone when the kernel restarts | Quick experiments and tests |
| `chromadb.PersistentClient(path="./chroma_db")` | A folder on your disk; survives restarts | Notebooks, prototypes, single-machine apps |
| `chromadb.HttpClient(host="localhost", port=8000)` | A Chroma server you start with `chroma run --path ./chroma_db` | Several apps or users sharing the same data |
| `chromadb.CloudClient(...)` | Chroma Cloud, a managed service | Production without running servers yourself |

The first two run **embedded** in your Python process (Chroma as a *vector store*). The last two talk to Chroma **over the network** (Chroma as a *vector database*). The collection methods (`add`, `query`, `get`, ...) are **the same for all four**, so moving from a notebook to a server means changing one line.

This is what a `PersistentClient` folder looks like on disk:

```
chroma_db/
├── chroma.sqlite3          ← ids, documents, metadata and collection settings
└── 987020e7-5722-.../      ← one folder per collection: its HNSW vector index
    ├── data_level0.bin
    ├── header.bin
    ├── length.bin
    └── link_lists.bin
```

The **HNSW** index is the graph-based ANN index from the previous notebook. Chroma builds and updates it for you.

## Embeddings: let Chroma do it, or bring your own

Chroma needs a vector for every record. There are three ways to provide it:

1. **Use the default model.** Pass only `documents=[...]` and Chroma embeds them with **`all-MiniLM-L6-v2`** (384 dimensions), running locally through ONNX. The first time, it downloads the model (about 80 MB) to `~/.cache/chroma/`. It's free, needs no API key, and works well for English text.
2. **Choose an embedding function.** Pass `embedding_function=...` when you create the collection, for example `OpenAIEmbeddingFunction` or `SentenceTransformerEmbeddingFunction` from `chromadb.utils.embedding_functions`. Chroma then calls that model whenever you add documents or query with text.
3. **Pass the vectors yourself.** Compute embeddings with any model (like the OpenAI or Hugging Face embeddings from section 02), then give them as `embeddings=[...]` when adding and `query_embeddings=[...]` when querying.

> **Important:** documents and questions must be embedded with the **same model**. Vectors from different models (or with different dimensions) can't be compared, so mixing them gives an error or meaningless results.

## Distance metrics

`query` returns **distances**, not similarities: the **smaller the distance, the more similar** the record.

| `space` | Measure | Formula | Notes |
|---|---|---|---|
| `l2` | Squared Euclidean distance | Σ (aᵢ − bᵢ)² | **The default** |
| `cosine` | Cosine distance | 1 − cosine similarity | The usual choice for text. 0 = same direction, 2 = opposite |
| `ip` | Inner product distance | 1 − (a · b) | For vectors that are already normalised |

The metric is set **when the collection is created** and can't be changed afterwards:

```python
collection = client.create_collection(
    name="research_papers",
    configuration={"hnsw": {"space": "cosine"}},
)
```

Since we used cosine similarity in section 02, `cosine` is the easiest to read: a distance of `0.2` means a cosine similarity of `0.8`.

For **normalised** embeddings (length 1, which includes the default MiniLM model and OpenAI embeddings), all three metrics put the results in the **same order**. Only the numbers differ.

## Working with a collection (CRUD)

| Method | What it does |
|---|---|
| `client.create_collection(name)` / `client.get_or_create_collection(name)` | Create a collection, or open it if it already exists |
| `collection.add(ids, documents, metadatas)` | Insert new records. If an id already exists, that record is **skipped**: the old one is kept |
| `collection.upsert(ids, documents, metadatas)` | Insert, or **overwrite** if the id already exists |
| `collection.update(ids, ...)` | Change the document, metadata or embedding of existing records |
| `collection.get(ids=..., where=...)` | Fetch records by id or by filter, like a normal database lookup (no similarity search) |
| `collection.query(query_texts, n_results, where=...)` | **Similarity search**: return the `n_results` closest records |
| `collection.delete(ids=..., where=...)` | Remove records |
| `collection.count()` / `collection.peek()` | Number of records / a quick look at the first few |
| `client.list_collections()` / `client.delete_collection(name)` | Manage collections |

The key difference to remember: **`get` finds records by id or exact filters**, while **`query` finds records by meaning**.

## Filtering with metadata and document text

`query`, `get` and `delete` can narrow down which records they look at.

**`where`** filters on **metadata**:

| Operator | Meaning | Example |
|---|---|---|
| `$eq`, `$ne` | equal / not equal | `{"source": "hr_policy.pdf"}` (short for `$eq`) |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than | `{"page": {"$gte": 3}}` |
| `$in`, `$nin` | in / not in a list | `{"year": {"$in": [2024, 2025]}}` |
| `$and`, `$or` | combine conditions | `{"$and": [{"source": "a.pdf"}, {"page": {"$lt": 10}}]}` |

**`where_document`** filters on the **document text**: `$contains`, `$not_contains`, `$regex` and `$not_regex`, which can also be combined with `$and` / `$or`.

```python
results = collection.query(
    query_texts=["How many vacation days do I get?"],
    n_results=3,
    where={"source": "hr_policy.pdf"},
    where_document={"$contains": "leave"},
)
```

This is the *"most similar chunks **where** `source = "hr_policy.pdf"`"* search described in the previous notebook: the filter decides **which** records are candidates, and similarity decides **how they are ranked**.

## Reading query results

`query` accepts a **list** of questions, so every field in the result is a **list of lists**: one inner list per question, sorted from most to least similar.

```
collection.query(query_texts=["What is attention?", "What is RAG?"], n_results=2)

{
  "ids":       [ ["chunk_7", "chunk_2"],          ["chunk_40", "chunk_41"]         ],
  "documents": [ ["Attention lets ...", "..."],   ["RAG combines ...", "..."]      ],
  "metadatas": [ [{...}, {...}],                  [{...}, {...}]                   ],
  "distances": [ [0.21, 0.35],                    [0.18, 0.29]                     ],
}                 └──── question 1 ────┘          └──── question 2 ────┘
```

So for a single question, the best match is `results["documents"][0][0]` and its distance is `results["distances"][0][0]`.

By default you get `documents`, `metadatas` and `distances`. Use `include=[...]` to choose the fields, for example add `"embeddings"` if you need the vectors themselves.

## Where Chroma fits in a RAG pipeline

```
INDEXING  (run once, and again whenever the documents change)

  documents ─► load & parse ─► chunk ─► collection.add(ids, documents, metadatas)
                                                    │  Chroma embeds each chunk
                                                    ▼
                                  ┌─────────────────────────────────┐
                                  │        Chroma collection        │
                                  │  HNSW vector index + SQLite     │
                                  └─────────────────────────────────┘
                                       ▲                       │
QUERYING  (every user question)        │ collection.query()    │ top-k chunks + metadata
                                       │                       ▼
  user question ───────────────────────┘     prompt = question + chunks ─► LLM ─► answer
```

The metadata that comes back with each chunk (file name, page, ...) is what lets the final answer **cite its sources**.

### With LangChain

LangChain wraps Chroma in its common `VectorStore` interface (package `langchain-chroma`), so it plugs straight into retrievers and chains:

```python
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,                    # LangChain Document objects
    embedding=embeddings,                # e.g. OpenAIEmbeddings() or HuggingFaceEmbeddings(...)
    collection_name="research_papers",
    persist_directory="./chroma_db",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

Here LangChain computes the embeddings with the model you pass and hands the vectors to Chroma (option 3 from the embeddings section).

## Strengths and limits

**Strengths**

- Zero setup: runs inside a notebook with one `pip install`
- Built-in embedding model, so you can start without any API key
- Persistence, CRUD and metadata / document filtering out of the box
- The same API from an in-memory experiment to a server or Chroma Cloud
- Open source (Apache 2.0)

**Limits to keep in mind**

- **Embedded mode is for one process.** Don't let several programs write to the same `PersistentClient` folder at once; run a Chroma server instead.
- **One machine, one memory.** A local or self-hosted single-node Chroma keeps its HNSW index in RAM, so very large collections need a bigger machine, Chroma Cloud, or a distributed vector database.
- **The default model is small.** `all-MiniLM-L6-v2` is fast but English-focused. For better retrieval, or other languages, use a stronger embedding model.
- **Pick the distance metric up front.** Changing it later means creating a new collection and re-adding the data.

## Key takeaways

- **Chroma** is an open-source vector database that starts as a Python library: `pip install chromadb`, no server needed.
- Data is organised as **client → collections → records**, and each record has an **id, document, embedding and metadata**.
- The **client** decides where data lives: `EphemeralClient` (memory), `PersistentClient` (local folder), `HttpClient` (server), `CloudClient` (managed). The collection API is the same for all of them.
- Chroma can **embed text for you** (default `all-MiniLM-L6-v2`) or use any model you choose. Always add and query with the **same model**.
- `query` returns **distances** (lower = more similar). The default metric is `l2`; set `cosine` when you create the collection if you want it.
- `where` and `where_document` combine **similarity search with filters** on metadata and text.
- LangChain's `Chroma` class plugs it straight into retrievers and RAG chains.

**Next:** we'll create a Chroma collection, add document chunks, and run similarity searches with filters.

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List

print(f"Import successful")

/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_35822/456648823.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Import successful


In [3]:


from pathlib import Path
import os

# Move to the project root, regardless of current notebook location
cwd = Path.cwd().resolve()
project_root = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists())

os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [4]:
## Document loading

from langchain_community.document_loaders import DirectoryLoader
doc_dir = r"./data/raw/text"
loader = DirectoryLoader(
    doc_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs= {'encoding':'utf-8'}
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"First Document preview : \n{documents[0].page_content[:200]}")

Loaded 2 documents
First Document preview : 
Machine Learning Algorithms: An Overview

Machine learning is a branch of artificial intelligence in which computers learn patterns from data instead of following hand-written rules. A machine learnin


In [5]:
## Document Splitting

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n"," "]
)

chunks = text_splitter.split_documents(documents=documents)
print(f"Created {len(chunks)} chunks")
print(f"Content : {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 19 chunks
Content : Machine Learning Algorithms: An Overview

Machine learning is a branch of artificial intelligence in which computers learn patterns from data instead ...
Metadata: {'source': 'data/raw/text/machine_learning_algorithms.txt'}


In [6]:
## Embedding the docuemtns
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x12d3d2cf0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x12d3d3a10>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [7]:
## Create embeddings for the chunks
texts = [chunk.page_content for chunk in chunks]
vectors = embeddings.embed_documents(texts)

print(f"Created {len(vectors)} embeddings")
print(f"Dimensions per vector: {len(vectors[0])}")
print(f"First 5 values of the first vector: {vectors[0][:5]}")


Created 19 embeddings
Dimensions per vector: 1536
First 5 values of the first vector: [-0.0242462158203125, -0.033935546875, 0.04559326171875, -0.025665283203125, 0.0196990966796875]


In [8]:
## Store the chunks in a Chroma vector store
persist_directory = "vectorstore/chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    collection_name="text_docs",
    persist_directory=persist_directory,
)

print(f"Stored {vectorstore._collection.count()} chunks in Chroma at {persist_directory}")


Stored 19 chunks in Chroma at vectorstore/chroma_db


In [9]:
## Similarity Search

query = " what are the types of neural netowrks?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs


[Document(metadata={'source': 'data/raw/text/deep_learning.txt'}, page_content='A neural network is made up of neurons organized into layers: an input layer, one or more hidden layers, and an output layer. Each connection between neurons has a weight. During training, the network makes predictions, measures its error with a loss function, and adjusts its weights using an algorithm called backpropagation combined with an optimizer such as gradient descent or Adam. Activation functions like ReLU, sigmoid, and tanh add non-linearity, which allows the network to learn complex'),
 Document(metadata={'source': 'data/raw/text/deep_learning.txt'}, page_content='Common Architectures\nConvolutional Neural Networks (CNNs) are designed for images. They use filters to detect edges, textures, and shapes, and are widely used in image classification and object detection.\n\nRecurrent Neural Networks (RNNs) and Long Short-Term Memory networks (LSTMs) process sequential data such as text, speech, and ti

In [10]:
## Similarity Search

query = " what is NLsupervised learningP?"

similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/raw/text/machine_learning_algorithms.txt'}, page_content='Supervised Learning\nIn supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).\n\nLinear Regression predicts a continuous value by fitting a straight line through the data. It is used for tasks such as predicting house prices or sales figures.'),
 Document(metadata={'source': 'data/raw/text/machine_learning_algorithms.txt'}, page_content='Reinforcement Learning\nIn reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward over time. Common algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimization (PPO). Reinforcement learning is used in g

In [11]:
print(f"Query: {query}")
print(f"\n Top {len(similar_docs)} similar chunks : ")
for i, doc in enumerate(similar_docs):
    print(f"\n --- Chunk {i+1} --- ")
    print(doc.page_content[:300] + "...")
    print(f"source : {doc.metadata.get('source','unknown')}")

Query:  what is NLsupervised learningP?

 Top 3 similar chunks : 

 --- Chunk 1 --- 
Supervised Learning
In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised tasks are either regression (predicting a number) or classification (predicting a category).

Linea...
source : data/raw/text/machine_learning_algorithms.txt

 --- Chunk 2 --- 
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward over time. Common algorithms include Q-Learning, Deep Q-Networks (DQN), and Proximal Policy Optimiz...
source : data/raw/text/machine_learning_algorithms.txt

 --- Chunk 3 --- 
K-Nearest Neighbors (KNN) classifies a new point by looking at the K closest points in the training data and taking a majority vote.

Naive Bayes uses Bayes' t

## How `similarity_search_with_score()` works

When you call:

```python
results_with_score = vectorstore.similarity_search_with_score(query=query, k=3)
```

this happens behind the scenes:

```
"What are the types of neural networks?"
                │
                ▼
 1. Embed the question        embeddings.embed_query(query)
                │              → a 1536-number vector (same model as the chunks)
                ▼
 2. Search the index          Chroma's HNSW index finds the stored chunk vectors
                │              closest to the question vector
                ▼
 3. Measure the distance      for each candidate: Σ (aᵢ − bᵢ)²   (squared L2)
                │
                ▼
 4. Keep the top k            sort by distance, smallest first, keep k = 3
                │
                ▼
 5. Return (Document, score)  chunk text + metadata, paired with its distance
```

**Step by step:**

1. **Embed the question.** LangChain turns the question into a vector using the **same** `OpenAIEmbeddings` model we used for the chunks. Vectors from the same model can be compared.
2. **Search the index.** Chroma doesn't compare the question with every chunk one by one. Its **HNSW** graph index jumps quickly to the closest area of the vector space. This is called *approximate nearest neighbour* (ANN) search, and it stays fast even with millions of chunks.
3. **Measure the distance.** For each candidate chunk, Chroma calculates the **squared Euclidean (L2) distance** between the question vector and the chunk vector. This is the default metric because we didn't choose one when creating the collection.
4. **Keep the top `k`.** The results are sorted by distance, and the `k` closest are kept.
5. **Return the results.** LangChain wraps each result as a `Document` (text + metadata) and pairs it with its distance, giving a list of `(Document, score)` tuples.

**Distance vs. cosine similarity**

OpenAI embeddings are normalised (length 1). For normalised vectors the L2 distance and the cosine similarity are directly linked:

$$\text{L2 distance} = 2 - 2 \times \text{cosine similarity}$$

| L2 distance (score) | Cosine similarity | Meaning |
|---|---|---|
| 0.0 | 1.0 | Identical meaning |
| 0.5 | 0.75 | Very similar |
| 1.0 | 0.5 | Somewhat related |
| 2.0 | 0.0 | Unrelated |
| 4.0 | −1.0 | Opposite |

So a **lower score means a better match**.

**The same search in plain Chroma**

`similarity_search_with_score()` is a thin LangChain wrapper around Chroma's own `query()`:

```python
query_vector = embeddings.embed_query(query)

raw = vectorstore._collection.query(
    query_embeddings=[query_vector],
    n_results=3,
)

raw["documents"][0]   # the chunk texts
raw["distances"][0]   # the same scores as similarity_search_with_score()
```


## Comparing the three search methods

LangChain's Chroma vector store has three ways to run a similarity search. All three do the **same search**: they embed the question, find the closest chunks in Chroma and return the top `k`. They differ only in **what score, if any, comes back with each chunk**.

| Method | Returns | Score meaning |
|---|---|---|
| `similarity_search(query, k)` | `[Document, ...]` | No score, just the chunks in best-first order |
| `similarity_search_with_score(query, k)` | `[(Document, distance), ...]` | Chroma's raw **distance**: **lower = better** |
| `similarity_search_with_relevance_scores(query, k)` | `[(Document, relevance), ...]` | A **relevance score**: **higher = better**, meant to be 0 to 1 |

### How they relate

```
similarity_search_with_score()                 → (Document, distance)
        │
        ├── similarity_search()                → drops the score → Document
        │
        └── similarity_search_with_relevance_scores()
                                               → converts distance → (Document, relevance)
```

- `similarity_search` calls `similarity_search_with_score` and throws the scores away.
- `similarity_search_with_relevance_scores` also calls `similarity_search_with_score`, then **converts each distance into a relevance score**. The formula depends on the collection's distance metric:

| Metric | Conversion LangChain uses |
|---|---|
| `l2` (the default) | `relevance = 1 - distance / √2` |
| `cosine` | `relevance = 1 - distance` (this is exactly the cosine similarity) |

**Example (`l2`):** a distance of `0.71` becomes `1 - 0.71 / 1.414 ≈ 0.50`.

### Filtering with a threshold

Relevance scores make it easy to drop weak matches:

```python
results = vectorstore.similarity_search_with_relevance_scores(
    query, k=3, score_threshold=0.4
)
```

Only chunks with a relevance score of `0.4` or higher are returned.

### A catch with the default `l2` metric

Chroma's `l2` is the **squared** distance (0 to 4 for normalised embeddings), but the `1 - d/√2` formula assumes the plain distance. Weak matches with a distance above about **1.41** therefore get **negative** relevance scores, and LangChain shows a warning: *"Relevance scores must be between 0 and 1"*.

To get clean 0 to 1 scores, create the collection with the **cosine** metric:

```python
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="text_docs_cosine",
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory=persist_directory,
)
```

The relevance score is then simply the cosine similarity.

### Which one to use

| Use | When |
|---|---|
| `similarity_search` | You only need the chunks, e.g. to pass them to the LLM |
| `similarity_search_with_score` | You want to see Chroma's raw distances |
| `similarity_search_with_relevance_scores` | You want an easy-to-read 0 to 1 score, or a `score_threshold` cutoff |


In [12]:
## similarity search with score
results_with_score = vectorstore.similarity_search_with_score(query=query, k=3)
results_with_score

vectorstore.similarity_search_with_relevance_scores

<bound method VectorStore.similarity_search_with_relevance_scores of <langchain_community.vectorstores.chroma.Chroma object at 0x12d70c6e0>>

In [13]:
## Generate an answer with an LLM using the retrieved chunks
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant that answers questions using ONLY the context provided.\n"
     "If the answer is not in the context, reply: \"I don't know based on the provided documents.\"\n"
     "Keep the answer clear and concise, and mention the source file(s) you used."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
])

query = "What are the types of neural networks?"
similar_docs = vectorstore.similarity_search(query, k=3)

context = "\n\n".join(
    f"[Source: {doc.metadata['source']}]\n{doc.page_content}" for doc in similar_docs
)

chain = prompt | llm
response = chain.invoke({"context": context, "question": query})

print(f"Question: {query}\n")
print(f"Answer:\n{response.content}")


Question: What are the types of neural networks?

Answer:
Convolutional Neural Networks (CNNs); Recurrent Neural Networks (RNNs); and Long Short-Term Memory networks (LSTMs). 
Source: data/raw/text/deep_learning.txt


## Modern RAG Chain

In [14]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the retrieved context. If the answer is not in the context, "
     "say you don't know based on the provided documents. Cite sources using the "
     "[Source: ...] labels included in the context. Keep the answer concise."),
    ("human", "Question: {input}\n\nContext:\n{context}"),
])

document_prompt = PromptTemplate.from_template(
    "[Source: {source}]\n{page_content}"
)
combine_docs_chain = create_stuff_documents_chain(
    llm,
    rag_prompt,
    document_prompt=document_prompt,
)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

query = "What are the types of neural networks?"
result = rag_chain.invoke({"input": query})

print(f"Question: {result['input']}\n")
print(f"Answer:\n{result['answer']}\n")
print("Retrieved sources:")
for source in dict.fromkeys(
    doc.metadata.get("source", "unknown") for doc in result["context"]
):
    print(f"- {source}")


Question: What are the types of neural networks?

Answer:
- Convolutional Neural Networks (CNNs) — designed for images
- Recurrent Neural Networks (RNNs) — for sequential data
- Long Short-Term Memory (LSTM) networks — a type of RNN for sequences

[Source: data/raw/text/deep_learning.txt]

Retrieved sources:
- data/raw/text/deep_learning.txt
